In [35]:
from typing import Literal

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from scipy.stats import chi2_contingency

In [ ]:
df = pd.read_csv("../dataframes/raw_data/loan_approval_data.csv")
df.head()

In [ ]:
df.info()

In [ ]:
len(df["Applicant_ID"].unique())

In [ ]:
df.isna().sum()

In [ ]:
df["Applicant_Income"].describe()

In [ ]:
sum(df.duplicated())

In [ ]:
numeric_columns = df.select_dtypes(include="number").columns.to_list()
cat_columns = df.select_dtypes(include="object").columns.to_list()

In [32]:
def hist_vizualizer(df: pd.DataFrame, cols: int = 2):
    rows = round(len(df.columns) / cols)

    fig, ax = plt.subplots(nrows=rows, ncols=cols, figsize=(10, 10))

    for i, col in enumerate(df.columns):
        r = i // cols
        c = i % cols
        ax[r, c].hist(df[col])
        ax[r, c].set_title(col)

    plt.tight_layout()
    plt.show()

In [ ]:
hist_vizualizer(df=df[numeric_columns])

In [ ]:
hist_vizualizer(df=df[cat_columns])

In [33]:
def boxplot_vizualizer(df, cols: int = 2):
    rows = round(len(df.columns) / cols)

    fig, ax = plt.subplots(nrows=rows, ncols=cols, figsize=(8, 16))

    for i, col in enumerate(df.columns):
        r = i // cols
        c = i % cols

        sns.boxplot(y=df[col], ax=ax[r, c])
        ax[r, c].set_title(col)

    plt.tight_layout()
    plt.show()

In [ ]:
boxplot_vizualizer(df=df[numeric_columns])

In [ ]:
def get_outliers_iqr(df, cols):
    limits = {}

    for col in cols:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1

        upper_outliers = Q3 + 1.5 * IQR
        lower_outliers = Q1 - 1.5 * IQR

        limits[col] = (upper_outliers, lower_outliers)
        outliers = df[col][(df[col] < lower_outliers) | (df[col] > upper_outliers)]

        print(f"\nCOLUMN: '{col}'")
        print(f"Upper limit of outliers: {upper_outliers}")
        print(f"Lower limit of outliers: {lower_outliers}")
        print(f"Amount of outliers: {len(outliers)}")

    return limits

In [ ]:
df_iqr = df.copy()
columns_with_outliers = []
limits = get_outliers_iqr(df=df_iqr, cols=df_iqr.select_dtypes("number").columns)

for col, (lower, upper) in limits.items():
    df_iqr[col] = df_iqr[col].clip(lower, upper)
    columns_with_outliers.append(col)

In [ ]:
boxplot_vizualizer(df_iqr[numeric_columns])

In [ ]:
hist_vizualizer(df=df_iqr[numeric_columns])

In [ ]:
df.columns

In [ ]:
df["Loan_Approved"] = df["Loan_Approved"].map({"No": 0, "Yes": 1})

In [ ]:
df["Applicant_Income"].describe()

In [ ]:
df["total_income"] = df["Applicant_Income"] + df["Coapplicant_Income"]

In [ ]:
df[["Loan_Amount", "Existing_Loans"]].corr()

In [ ]:
df["approximate_loan_amount"] = df["Loan_Amount"] / (df["Existing_Loans"] + 1)

In [ ]:
df["DTI_Ratio"].describe()

In [ ]:
df["payment_capacity"] = ((df["total_income"]) + df["Savings"]) / (
    df["Loan_Amount"] * (df["Existing_Loans"] + 1) / (df["Credit_Score"] + 1)
)

In [ ]:
df["collateral_ratio"] = df["Collateral_Value"] / (df["Loan_Amount"] + 1)

In [ ]:
df["income_to_loan"] = (df["total_income"]) / (df["Loan_Amount"] + 1)

In [ ]:
df["risk_score"] = (df["DTI_Ratio"] * df["Loan_Amount"]) / (df["Credit_Score"] + 1)

In [ ]:
df.drop(["Applicant_ID"], inplace=True, axis=1)

In [ ]:
df.to_csv("../dataframes/prepared/loan_feature_extracted.csv", index=False)